# Archive 앞부분 조사·선택 추출 — 위에서 아래로 1회

**소스 ZIP 업로드 → 로컬 검사 → JOB 한 곳 설정 → 네트워크 없는 preflight → 명시적 1회 실행 → 결과 검토 → ZIP 보관** 순서입니다. 별도 plan/선택 JSON 업로드나 아래쪽 새 job 셀은 없습니다. 사용자용 사본은 JOB만 미리 채워 제공할 수 있습니다.

기본 `scan`은 실제 Drive에서 byte0부터 제한된 압축 구간을 읽고 해제하여 regular member의 순서·이름·크기를 기록합니다. **Payload 저장0이지만 네트워크 비용은 발생합니다.** Gzip 임의 위치 점프·이어받기·전체 목록/색인 생성이 아닙니다. `includes`는 저장/선택 필터이며 압축 위치를 바꾸지 않습니다. 같은8MiB를 반복해도 다음8MiB로 전진하지 않습니다.

기본 smoke8MiB_v1은 본문8MiB·60초·disk16MiB·요청128입니다. 별도 counterpart64MiB_v1은 본문64MiB·300초·요청768·해제128MiB, 저장2파일의 고정 예산이며 mode=extract/purpose=targeted와 exact_targets2개를 요구합니다. 두 프로파일 모두 Range256KiB·read1/해제64KiB·disk16MiB를 유지합니다. Cache 없는 경로는 byte0부터 원격으로 읽습니다. Cache profile은 검증된 prefix를 byte0부터 로컬 재해제하고 미보관 suffix만 원격으로 읽으며 gzip 내부 상태를 이어받는 기능은 아닙니다. Source 전후 검사를 통과하지 못하면 목록도 untrusted입니다. 모든 결과는 train_ready=false입니다. [준비 계약](../docs/ARCHIVE_PREPARATION.md), [실행 안내](../docs/COLAB_SETUP.md)를 따릅니다.

Pose 전용 `poseCounterpart64MiB_v1`은 kind=pose·extract/targeted·정확한2목표를 요구합니다. Body64MiB/300초/768요청은 유지하고 header10000·누적tar metadata1MiB·해제512MiB로 구분합니다. 단일 GNU/PAX metadata는64KiB이며 파일4MiB/disk16MiB/Range256KiB/내부IO64KiB는 그대로입니다. 성공/목표위치/전체목록을 보장하지 않습니다.

2026-10-27 사용자 결정으로 실제 Colab 테스트는 일시 보류합니다. 아래는 재개 시 참조용입니다. 재개 전 테스트 세션에서 고정 revision·소스 ZIP·ready notebook·cache 식별값과 실행 조건을 확인하며 이전 배포본을 문서 갱신으로 교체하지 않습니다. [현재 결정](../docs/REAL_DATA_NEXT_RUN.md)을 따릅니다.


In [ ]:
import sys, os, io, stat, json, tempfile, zipfile, shutil
from pathlib import Path, PurePosixPath
from google.colab import files

print({"python":sys.version.split()[0],"free_disk_bytes":shutil.disk_usage(".").free})
uploaded = files.upload()  # 여기에는 코드가 들어 있는 source ZIP 한 개만 올립니다.
if len(uploaded)!=1:
    raise ValueError("코드 source ZIP 한 개만 선택하세요.")
source_name, source_bytes = next(iter(uploaded.items()))
if not source_name.lower().endswith(".zip") or len(source_bytes)>64*1024**2:
    raise ValueError("64MiB 이하의 코드 source ZIP이 필요합니다.")
excluded={".git",".venv","data","outputs","checkpoints",".aws",".codex",".agents","__pycache__"}
with zipfile.ZipFile(io.BytesIO(source_bytes)) as archive:
    members=archive.infolist()
    if len(members)>5000 or sum(m.file_size for m in members)>128*1024**2:
        raise ValueError("코드 ZIP의 항목 수/해제 크기 상한을 초과했습니다.")
    code_entries=[m for m in members if m.filename.endswith("rewind_archive/__init__.py")]
    if len(code_entries)!=1:
        inner=[PurePosixPath(m.filename).name for m in members if m.filename.lower().endswith(".zip")][:6]
        if inner:
            raise ValueError("배포 묶음 ZIP입니다. 로컬에서 내부 코드 ZIP "+repr(inner)+"만 꺼내 다시 올리세요. 중첩 자동 해제는 하지 않습니다.")
        raise ValueError("rewind_archive/__init__.py를 포함한 코드 source ZIP이 필요합니다. Notebook/결과 ZIP과 구분하세요.")
    seen=set()
    for m in members:
        p=PurePosixPath(m.filename)
        if p.is_absolute() or ".." in p.parts or "\\" in m.filename or any(":" in x for x in p.parts) or any(ord(c)<32 for c in m.filename) or stat.S_ISLNK(m.external_attr>>16):
            raise ValueError("안전하지 않은 코드 ZIP 경로입니다.")
        key=p.as_posix().casefold().rstrip("/")
        if key in seen:raise ValueError("중복 코드 ZIP 경로입니다.")
        seen.add(key)
    extract_root=Path(tempfile.mkdtemp(prefix="rewind_source_",dir="/content"))
    for m in members:
        parts=PurePosixPath(m.filename).parts
        if not any(p in excluded or p==".env" or p.startswith(".env.") for p in parts):
            archive.extract(m,extract_root)
roots=[p.parent.parent for p in extract_root.rglob("rewind_archive/__init__.py")]
if len(roots)!=1:raise ValueError("단일 프로젝트 source ZIP이 필요합니다.")
PROJECT_ROOT=roots[0]
if not (PROJECT_ROOT/"rewind_archive/workflow.py").is_file():
    raise ValueError("이 notebook과 맞는 최신 source ZIP이 필요합니다(workflow.py 누락).")
if any(name=="rewind_archive" or name.startswith("rewind_archive.") for name in sys.modules):
    raise RuntimeError("이전 모듈이 이미 import되어 있습니다. 기존 결과를 먼저 보관하고 런타임을 재시작한 뒤 새 source ZIP으로 시작하세요.")
os.chdir(PROJECT_ROOT)
sys.path.insert(0,str(PROJECT_ROOT))
from rewind_archive.workflow import NOTEBOOK_API_VERSION
if NOTEBOOK_API_VERSION!=9:raise RuntimeError("Notebook/source API 버전이 다릅니다.")
stamp=PROJECT_ROOT/"SOURCE_REVISION.txt"
print({"source_root":str(PROJECT_ROOT),"source_revision":stamp.read_text().strip() if stamp.exists() else "unversioned; code hashes recorded","notebook_api":NOTEBOOK_API_VERSION})
del uploaded, source_bytes


## 로컬 사전 검사

API9 소스의 사전 검사153개에 대한 안내입니다. 합성 archive·모의 Drive·localhost 검사입니다. 실제 Google 인증·원본 요청은 하지 않습니다. 실패하면 실제 실행으로 진행하지 않습니다. 이 단계는 모델 학습/GPU 시험과 다릅니다.


In [ ]:
import subprocess
RUN_LOCAL_CHECKS = True
if RUN_LOCAL_CHECKS:
    for pattern in ("test_archive.py","test_drive_metadata.py","test_archive_planning.py","test_archive_scan.py","test_archive_workflow.py","test_archive_targets.py","test_archive_pose_profile.py","test_archive_request_diagnostics.py","test_archive_request_retry.py","test_archive_prefix_cache.py","test_archive_warm128.py","test_archive_warm192.py"):
        subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-p",pattern,"-v"],cwd=PROJECT_ROOT,check=True)


## 1. 사용자 설정 — 아래 JOB만 편집

캐시 없는 기본 조사 예시는 `mode="scan"`, `purpose="prefix_inventory"`, `includes=[]`입니다. 확인한 RGB000의 selection을 채우거나 이미 채운 사용자용 notebook을 사용합니다. 별도 JSON 업로드는 필요 없습니다. Scan은 최대128개 regular header를 기록하고 payload는 저장하지 않습니다.

이번 대응 추출은 `profile="counterpart64MiB_v1"`, `mode="extract"`, `purpose="targeted"`와 exact_targets의 정확한 경로2개를 사용합니다. Includes도 같은 두 경로여야 합니다. 프로파일명과 실제 limits가 preset과 다르면 인증 전에 거부하며 임의 수치 확대는 허용하지 않습니다. 두 목표가 payload+padding+저장 완료되면 즉시 중단합니다. 캐시 없는 profile은 실제 관측 경로를 입력해도 byte0부터 원격으로 다시 읽습니다. 캐시 profile은 byte0 로컬 재해제 후 미보관 suffix만 원격으로 읽습니다. 기존 custom8MiB discovery는 profile=None, max_files2이하의 이전 계약을 유지합니다. `review`/`review`는 previous_result의 기존 런타임 결과만 검사하며 네트워크를 사용하지 않습니다. 저장한 결과 ZIP을 자동 업로드/풀지는 않습니다.

같은 job_id는 실패해도 다시 실행할 수 없습니다. 같은 설정 fingerprint를 다시 읽으려면 **이 JOB에서 새 job_id와 변경한 reason**을 명시해야 합니다. 설정 변경 후에는 아래 preflight부터 순서대로 실행합니다. 커널 밖 과거 실행 기록까지 자동 인식하지 못하므로 이전 ZIP과 기록도 확인하세요. ID/인증정보를 저장소에 커밋하지 않습니다.

API9 JOB의 request_policy 기본값은 None(기존5초·추가시도0)입니다. poseCounterpart64MiB_v1, posePrefixCache96MiB_v1, posePrefixWarm128MiB_v2 또는 posePrefixWarm192MiB_v3의 targeted 경로2개에서 metadataPreTimeout10sRetry1Global3_v1을 선택할 수 있습니다. 정책은 preflight와 fingerprint에 포함되고 초기 metadata 별도예산은 유지합니다.

API7의 posePrefixCache96MiB_v1은 단일posepart/정확목표2개 전용입니다. cache_input=None은 cold, 이후 별도 고정JOB에서 cache_input="upload"는 보관cache ZIP 한 개를 재사용합니다. 원본 반복은 자동 실행하지 않습니다. cache의 준비완료와 목표확보는 별도로 판단합니다. 자세한 계약은 [캐시 안내](../docs/PREFIX_CACHE.md)를 따릅니다.

API8: posePrefixWarm128MiB_v2는 cache 입력과 고정 cache_expectation(manifest_sha256, prefix_sha256, prefix_bytes)이 필수입니다. 입력이 없거나 다르면 cold로 전환하지 않습니다. Ready notebook은 검토된 입력 식별값을 고정합니다. 구형 JOB도 계속 지원합니다.

API9: posePrefixWarm192MiB_v3는128MiB 이상의 고정 cache가 필수입니다. cache_expectation과 검토된 ZIP을 사용하며 cold fallback은 없습니다. 이전 profile 계약은 유지합니다. 고정128MiB 입력에서 새 media는 최대64MiB이며 논리192MiB·새 응답body80MiB·준비420초·773HTTP입니다. byte0부터 캐시를 로컬 재해제하고 미보관 suffix만 원격으로 읽습니다. 사전 검사의 큰 합성 fixture는 임시 폴더에서 정리하며 준비420초/단계disk 예산과 별도입니다.


In [ ]:
from dataclasses import asdict
from rewind_archive.profiles import profile_limits
from rewind_archive.drive_metadata import MetadataLimits

BUDGET_PROFILE = "smoke8MiB_v1"  # 별도 fixed targeted: counterpart64MiB_v1 / poseCounterpart64MiB_v1
JOB = {
    "job_id":"prefix_scan_001",
    "reason":"archive 앞부분 regular member 순서 조사",
    "mode":"scan",
    "purpose":"prefix_inventory",
    "profile":BUDGET_PROFILE,
    "cache_expectation":None,  # warm128: pinned manifest_sha256 / prefix_sha256 / prefix_bytes
    "cache_input":None,  # warm: "upload" (cache ZIP one file); cold: None
    "request_policy":None,  # pose targeted만: metadataPreTimeout10sRetry1Global3_v1
    "selection":{
        "schema_version":"drive_part_selection_v1",
        "archive_id":"", "dataset_version":"", "kind":"rgb", "total_parts":None,
        "parts":[]  # [{"index":0,"name":"USER_ARCHIVE.tar.gz000","file_id":"USER_FILE_ID"}]
    },
    "includes":[],
    "exact_targets":[],  # targeted: glob 없는 canonical 경로; includes와 동일한 집합
    "limits":asdict(profile_limits(BUDGET_PROFILE)),
    "metadata_limits":asdict(MetadataLimits()),
    "previous_result":""  # mode/purpose=review일 때 기존 런타임 결과 폴더
}


## 2. 최종 preflight — 원본 요청·인증 없음

Mode/purpose·profile·selection/archive·includes/exact_targets·body/time/requests/decompressed/disk·Range/IO와 동일 작업 경고를 확인합니다. 두 단계의 상한 합계와 비용 제외 범위도 표시합니다. 선택한 profile의 준비 예산(smoke8MiB/60초, counterpart64MiB/300초, cache/warm420초)과 최초metadata128KiB/30초는 별도이며 인증/전체wire 비용은 이 합계 밖입니다. 이전 실제8MiB 관측의 단순 비례 추정은 약234초이나 성공·실행 시간·hard deadline을 보장하지 않습니다. 표시한 설정을 아래 실행이 그대로 사용합니다. 이 셀 뒤에서 JOB를 바꾸면 실행을 거부합니다. Size/version/modifiedTime은 아직 조회하지 않았으며 실제 metadata 성공 후 media 직전에 별도로 표시합니다.

이력이 없는 것이 과거 실행이 없었다는 증거는 아닙니다. 현재 커널 기록과 현재 소스 outputs의 job 기록만 검사합니다. 같은 설정/선택을 읽은 이력이 표시되면 원인·목록 조사 목적·예산을 다시 검토합니다.


In [ ]:
from rewind_archive.workflow import make_preflight, load_history
_ARCHIVE_JOB_HISTORY = globals().get("_ARCHIVE_JOB_HISTORY",{})
_ARCHIVE_JOB_HISTORY.update(load_history(PROJECT_ROOT/"outputs"))
if JOB.get("cache_input")=="upload":
    from rewind_archive.cache_policy import for_profile
    policy=for_profile(JOB.get("profile"))
    expected=JOB.get("cache_expectation")
    placeholder={"path":"pending_upload","manifest_sha256":expected["manifest_sha256"]} if policy and expected else None
    make_preflight({**JOB,"cache_input":placeholder},_ARCHIVE_JOB_HISTORY)
    from rewind_archive.prefix_cache import save_uploaded_cache
    JOB["cache_input"],CACHE_UPLOAD_REPORT=save_uploaded_cache(files.upload(),PROJECT_ROOT/"cache_inputs",
        policy=policy,expectation=expected)
    print({"cache_upload":CACHE_UPLOAD_REPORT})
PREFLIGHT = make_preflight(JOB,_ARCHIVE_JOB_HISTORY)
print(json.dumps(PREFLIGHT["summary"],ensure_ascii=False,indent=2))
print({"config_sha256":PREFLIGHT["config_sha256"],"effective_sha256":PREFLIGHT["effective_sha256"],"network_requests":0})


## 3. 명시적 1회 실행 — 기본 비활성

위 요약을 검토한 뒤 **EXECUTE_ONCE=True**로 이 셀을 한 번 실행합니다. Preflight 설정이 현재 JOB와 다르면 인증 전 거부합니다. 시작한 job은 먼저 기록한 뒤 인증→metadata→선택한 고정 프로파일의 bounded scan/extract 순서로 진행합니다. JOB 자체를 자동 재실행하지 않습니다. 명시한 opt-in 정책에 한해 사전 metadata 개시 timeout을 제한 재시도합니다.

최초Metadata128KiB/30초는 선택한 준비 예산(smoke60초, counterpart/pose-counterpart300초, cache/warm420초)과 별도이며 전후 metadata 비용은 준비 예산에 포함됩니다. 최초 인증 비용·전체 wire/OS/RSS는 측정 밖입니다. Scan도 앞 payload를 읽고 해제하는 비용을 냅니다. Name/glob은 seek가 아닙니다. 캐시 없는 새 JOB는 byte0부터 원격 재읽기이며, 검증된 cache를 쓰는 JOB는 byte0부터 로컬 재해제하고 미보관 범위만 원격으로 읽습니다. 실행 뒤에는 아래 결과 보관으로 진행합니다.


In [ ]:
EXECUTE_ONCE = False
from rewind_archive.workflow import run_workflow_job
if JOB["mode"]=="review" or EXECUTE_ONCE:
    current=make_preflight(JOB,_ARCHIVE_JOB_HISTORY)
    if current["config_sha256"]!=PREFLIGHT["config_sha256"]:
        raise RuntimeError("JOB 설정이 바뀌었습니다. Preflight부터 다시 실행하세요.")
if JOB["mode"]=="review":
    # Review 설정도 preflight와 동일해야 한다. 원본/인증 호출은 없다.
    RESULT_ROOT=Path(JOB["previous_result"])
elif EXECUTE_ONCE:
    print("실행 직전 effective 설정:",json.dumps(PREFLIGHT["summary"],ensure_ascii=False,indent=2))
    def create_drive_session():
        from google.colab import auth
        import google.auth
        from google.auth.transport.requests import AuthorizedSession, Request
        from requests.adapters import HTTPAdapter
        import urllib3
        if not callable(getattr(urllib3.response.HTTPResponse,"read1",None)):
            raise RuntimeError("urllib3 raw.read1 API가 필요합니다.")
        auth.authenticate_user()
        credentials,_=google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
        # opt-in에서는 인증용 HTTP session도 숨은 retry0으로 고정한다.
        auth_http = None
        if JOB.get("request_policy") or JOB.get("profile") in ("posePrefixCache96MiB_v1","posePrefixWarm128MiB_v2","posePrefixWarm192MiB_v3"):
            import requests
            auth_http=requests.Session()
            auth_http.trust_env=False
            auth_http.mount("https://",HTTPAdapter(max_retries=0))
        try:
            auth_request=Request(session=auth_http) if auth_http is not None else Request()
            credentials.refresh(auth_request)
            session=AuthorizedSession(credentials,max_refresh_attempts=0,auth_request=auth_request) if auth_http is not None else AuthorizedSession(credentials,max_refresh_attempts=0)
            if auth_http is not None:
                # custom auth_request session은 AuthorizedSession.close가 소유하지 않는다.
                original_close=session.close
                def close_sessions():
                    try:original_close()
                    finally:auth_http.close()
                session.close=close_sessions
            session.trust_env=False
            session.mount("https://",HTTPAdapter(max_retries=0))
        except BaseException:
            if auth_http is not None:auth_http.close()
            raise
        return session
    RESULT_ROOT, JOB_RESULT = run_workflow_job(JOB,PREFLIGHT,_ARCHIVE_JOB_HISTORY,PROJECT_ROOT/"outputs",create_drive_session,
        cache_upload_report=globals().get("CACHE_UPLOAD_REPORT") if JOB.get("cache_input") else None)
    print({"result_root":str(RESULT_ROOT),"status":JOB_RESULT["status"],"stop_reason":JOB_RESULT.get("stop_reason"),"train_ready":False})
else:
    print("실제 인증·metadata·media를 실행하지 않았습니다.")


## 4. 결과 검토 — 원본 요청 없음

`scan`의 저장 파일0개는 의도한 동작입니다. 관측목록의 header/regular ordinal은 tar 안에서 도달한 순서이며 frame 순서가 아닙니다. `payload_complete`는 payload+padding을 소비했다는 뜻이고 `stored`와 다릅니다. Name/size는 header 선언이며 PNG/JSON 내용 검증이 아닙니다. 불완전 payload는 별도로 표시합니다.

`bounded_observation_only`도 전체 목록/seek offset 증명이 아닙니다. Source/unsafe/parser 실패 시 `untrusted_failed_source_or_parser`이므로 목록에서 후속 파일 확보를 확정하지 않습니다. Partial·상한·CRC·목록 cap을 함께 확인합니다. Exact targets를 지정했다면 completed_targets/missing_targets/targets_complete/targets_status를 확인합니다. `stop_reason="targets_complete"`는 두 목표 저장에 따른 추출 중단 이유입니다. 이 값만으로 hash 검증 성공을 뜻하지 않습니다. 최종 성공은 pilot verification.status=passed, targets_complete=true, targets_status=complete 및 현재 파일 검증 통과를 함께 확인합니다. 전체 gzip/tar 완료는 별도입니다. 다음 header/Range는 읽지 않지만 이미 받은 Range의 선수신 bytes는 비용에 포함됩니다. 실패한 원본/파일 검증이나 잘린 목표는 completed로 세지 않습니다. 전부 못 찾으면 상한·오류·미확보 목록을 보고하고 정지합니다. 성공/미확보/목록 조사 모두 ZIP을 보관하고 검토를 위해 중단합니다. 다음 archive·반대쪽 추출을 자동 시작하지 않습니다.

최초128개 목록과 별도로 `last_regular_observation` 한 행과 `regular_members_seen` 합계가 마지막 도달 regular header를 보여줍니다. Payload 완료/저장 여부와 전체 trust를 함께 확인합니다. Last 행은 불완전할 수 있으며 원본/후검사/CRC/parser 실패이면 이 행도 진단용 untrusted입니다. Header ordinal은 정확한 파일 seek offset이 아닙니다.

요청 실패 진단은 request_failure의 category/library/operation/stage만 확인합니다. HTTP response 분류와 다른 구조이며 status 미기록은 HTTP 상태를 모른다는 뜻입니다. 알 수 없는 예외는 unknown이고, 원문·URL·token·header·body·임의 클래스명은 출력하지 않습니다. 응답 body/검증 실패 뒤 cleanup 오류는 첫 실패를 덮지 않으며, 진단이 없는 이전 보고서는 request_failure_recorded=false로 표시합니다. 과거 실패 원인을 새 코드로 소급 추정하지 않습니다.

request_retry는 최종 request_failure와 별개입니다. 성공해도 앞선 timeout 이력·실제 추가시도 수·대기시간을 남기며, exhausted/decision으로 시도 상한과 요청·시간·전송 예산 차단을 구분합니다. pre 개시는 최대10초(남은 시간으로 축소), 같은 pre당 추가1회·전체 추가3회·재시도 전0.5초이고 선택한 profile의 전체 준비 예산(pose-counterpart300초, cache/warm420초)에 포함합니다. 응답을 받은 HTTP403/429·부분 본문·media·후검사·원본 변경은 재시도하지 않습니다. socket/cooperative 제한이며 DNS/OS hard deadline은 보장하지 않습니다.


In [ ]:
from rewind_archive.workflow import review_run
if globals().get("RESULT_ROOT") is not None:
    REVIEW = review_run(RESULT_ROOT)
    # 긴 metadata 응답별 audit는 ZIP 보고서에 유지하고 화면에는 합계만 표시한다.
    if "extraction" in REVIEW:
        REVIEW["extraction"].get("stats",{}).pop("metadata_response_audit",None)
    print(json.dumps(REVIEW,ensure_ascii=False,indent=2))
else:
    print("새 결과가 없습니다. 기존 결과를 보려면 단일 JOB의 mode/purpose=review와 previous_result를 설정하세요.")


## 5. 결과 ZIP 저장·다운로드

현재 작업의 job/metadata/pilot/관측목록·완전 파일만 작은 ZIP으로 보관합니다. 원본이나 소스 전체를 넣지 않습니다. 최대17MiB 결과와 비슷한 별도 임시 ZIP 공간을 사용합니다. 기존 ZIP은 덮어쓰거나 자동 추가하지 않으며 다운로드만 다시 실행할 수 있습니다. 개인 file ID/sample이 포함될 수 있으므로 개인 보관합니다. **런타임을 종료하기 전에 ZIP을 내려받으세요.**

API7/8/9 cache는 결과ZIP과 별도이며 cache export를 먼저 시도합니다. 실패해도 결과ZIP은 보관할 수 있습니다. DOWNLOAD_CACHE_ZIP도 기본False입니다. Cache ZIP과 결과ZIP의 경로/hash를 각각 보관하고,기존ZIP 재다운로드는 동일snapshot을 유지합니다. Warm시험은 별도고정JOB/누적예산으로 인계합니다.


In [ ]:
DOWNLOAD_RESULT_ZIP = False
import hashlib
DOWNLOAD_CACHE_ZIP = False
from rewind_archive.workflow import save_result_zip, save_cache_zip
from rewind_archive.contracts import PreparationError
if globals().get("RESULT_ROOT") is not None:
    root=Path(RESULT_ROOT)
    pilot=root/"pilot" if (root/"pilot").is_dir() else root
    # Cache export first; result ZIP remains downloadable even if export fails.
    if (pilot/"cache/manifest.json").is_file():
        try:
            CACHE_ZIP_PATH,CACHE_EXPORT_REPORT=save_cache_zip(root)
            print({"cache_zip":str(CACHE_ZIP_PATH),"cache_export":CACHE_EXPORT_REPORT})
            if DOWNLOAD_CACHE_ZIP:files.download(str(CACHE_ZIP_PATH))
        except (PreparationError,OSError,ValueError) as exc:
            print({"cache_export_failed":exc.reason if isinstance(exc,PreparationError) else "cache_export_io_or_report_failed"})
    prior_zip=root.with_suffix(".zip").exists()
    RESULT_ZIP_PATH=save_result_zip(root)
    print({"result_zip":str(RESULT_ZIP_PATH),"result_zip_sha256":hashlib.sha256(RESULT_ZIP_PATH.read_bytes()).hexdigest(),
        "existing_result_zip_reused":prior_zip})
    if DOWNLOAD_RESULT_ZIP:files.download(str(RESULT_ZIP_PATH))
else:
    print("No result yet.")
